## __Phase 4c: Open-set evaluation (Phase 2)__

Evaluates whether the matching pipeline can correctly reject RF points that have no
true matching AIS track, instead of always force-picking a best candidate. Reuses the
same prefilter/scoring functions and `scripts/evaluation/metrics.py`, but drives them
without the closed-set bypass.

The primary negative class here is a **dark-vessel** split: the vessel universe is
partitioned once into a registry pool (the only vessels ever offered as a search
candidate) and a held-out dark pool (queried, but never searchable by anyone). Unlike
leave-one-out (which only ever excludes a query's own track, leaving every other
vessel -- dark-designated or not -- in the candidate pool), a dark vessel here is never
a candidate for anyone, so nothing about it leaks into any other result. See
`scripts/evaluation/open_set.build_dark_vessel_frame` for why this can be built by
filtering the already-computed closed-set data instead of a fresh alignment run, and
`scripts/evaluation/run_phase4c_open_set_evaluation.py` for the full evaluation (2a
prefilter-recall diagnostic, dark-vessel ROC/PR/AUC, combined Phase 1+2 summary, plus a
secondary leave-one-out comparison where that data exists):

```bash
python -m scripts.evaluation.run_phase4c_open_set_evaluation --error-model uniform
```

This notebook keeps a light interactive version for exploration only: it imports that
script's own compute functions (nothing duplicated here) and plots the overall ROC/PR
curves live via `scripts.evaluation.plots`.

In [ ]:
import pandas as pd

from scripts.evaluation import plots
from scripts.evaluation.open_set import (
    MODELS,
    build_dark_vessel_frame,
    compute_pr,
    compute_roc,
    load_closed_results,
    split_registry_dark_vessels,
)

In [ ]:
# Select which RF bearing-error model's data to evaluate (must match the
# --error-model used to generate/run the upstream scripts). Unlike the
# leave-one-out negatives, the dark-vessel split only needs the closed-set
# data, so this works for either error model without extra pipeline runs.
ERROR_MODEL = "uniform"  # "uniform" or "gaussian"
DATA_DIR = f"../data/processed/{ERROR_MODEL}"

df_preselection = pd.read_pickle(f"{DATA_DIR}/AIS_RF_preselection_data.pkl")
closed_results = load_closed_results(DATA_DIR)

vessel_ids = pd.concat([df_preselection["RF_track_id"], df_preselection["AIS_track_id"]]).unique()
registry_ids, dark_ids = split_registry_dark_vessels(vessel_ids)
print(f"Vessel universe: {len(vessel_ids)}; registry: {len(registry_ids)}; dark (held out): {len(dark_ids)}")

### __Overall ROC + Precision-Recall, one curve per model (dark-vessel negatives)__

In [ ]:
dark_vessel_frames = {
    name: build_dark_vessel_frame(name, df_preselection, closed_results, registry_ids, dark_ids, score_col, mode)
    for name, _, _, score_col, mode in MODELS
}

overall_curves = {
    name: {**compute_roc(frame, "accept_score"), **compute_pr(frame, "accept_score")}
    for name, frame in dark_vessel_frames.items()
}
plots.roc_pr_overall(overall_curves, negatives_label="all dark-vessel negatives")